# Fundamentals 04 - Human Result API

Objetivo: aprender las vistas publicas de un `RunResult` real. El notebook no construye payloads de respuesta ni fija el resultado esperado.

## Parametros de la demostracion

| Parametro | Default | Proposito |
|---|---|---|
| symbol | RunResult | Entrada de una ejecucion real. |
| view | human, output, summary | Comparar representaciones del mismo RunResult. |
| serialization | toolkit.show_json | Aplicar el contrato polimorfico publico. |

## 1) Producir un RunResult

La ejecucion local es determinista, pero recorre Tool, Runtime, System y Agent reales.

In [ ]:
import agentic_systems as toolkit

runtime = toolkit.runtime(provider="python-runtime")
system = toolkit.system(runtime=runtime)

@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {
        "symbol": symbol,
        "is_public": symbol in toolkit.__all__,
        "package_version": toolkit.__version__,
    }

agent = system.agent(
    name="public_api_inspector",
    instructions="Ejecuta inspect_public_api y conserva la evidencia observada.",
    tools=[inspect_public_api],
    runtime=runtime,
    contract=toolkit.AgentContract(must_call=["inspect_public_api"]),
    policy=toolkit.RunPolicy(max_tool_calls=1, max_turns=2, temperature=0.0),
)

result = agent.run(
    {"tool": "inspect_public_api", "input": {"symbol": "RunResult"}},
    mode="eval",
)
assert result.ok

## 2) Elegir la vista adecuada

- `human_result`: lectura semantica.
- `run_result_output`: contrato de auditoria.
- `run_result_summary`: resumen compacto.
- `run_result_view`: vista para exploracion.

In [ ]:
toolkit.human_result(result, title="Human RunResult", show_lineage=True)
toolkit.show_json(toolkit.run_result_output(result), title="run_result_output")
toolkit.show_json(toolkit.run_result_summary(result), title="run_result_summary")
toolkit.show_json(toolkit.run_result_view(result), title="run_result_view")

## 3) Polimorfismo estructural

`show_json` acepta Pydantic-like objects, `to_dict`, dataclasses, mappings y secuencias sin conocer providers.

In [ ]:
from dataclasses import dataclass

@dataclass
class ReviewNote:
    symbol: str
    accepted: bool

note = ReviewNote(
    symbol=result.data["symbol"],
    accepted=bool(result.data["is_public"]),
)
toolkit.show_json(note, title="Dataclass derivada del RunResult")

## 4) API realmente ejercitada

In [ ]:
api_coverage = [
    "toolkit.runtime", "toolkit.system", "toolkit.tool", "system.agent", "agent.run",
    "toolkit.human_result", "toolkit.run_result_output", "toolkit.run_result_summary",
    "toolkit.run_result_view", "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="Human result API coverage")

## Resultado esperado

Todas las vistas describen la misma ejecucion y conservan `RunResult` como fuente unica de verdad.